In this assignment i would be using the red wind dataset. It contains chemical measurements of wine, such as acidity, sugar, alcohol, and pH, along with a quality score based on sensory evaluation.
I would be using Machine learning models to learn patterns between the chemical measurements and these quality groups, then predict the quality group of wines in the test set.

In [1]:
#import the necessary libraries.
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

In [2]:
#load the dataset
df = pd.read_csv("winequality-red.csv")

In [3]:
#Explore the dataset
df.head()


,"fixed acidity;""volatile acidity"";""citric acid"";""residual sugar"";""chlorides"";""free sulfur dioxide"";""total sulfur dioxide"";""density"";""pH"";""sulphates"";""alcohol"";""quality"""
0,7.4;0.7;0;1.9;0.076;11;34;0.9978;3.51;0.56;9.4;5
1,7.8;0.88;0;2.6;0.098;25;67;0.9968;3.2;0.68;9.8;5
2,7.8;0.76;0.04;2.3;0.092;15;54;0.997;3.26;0.65;...
3,11.2;0.28;0.56;1.9;0.075;17;60;0.998;3.16;0.58...
4,7.4;0.7;0;1.9;0.076;11;34;0.9978;3.51;0.56;9.4;5


In [4]:
#reload the dataset specifying that it should be separated into columns whereever there is a ; separator
df = pd.read_csv("winequality-red.csv", sep=";")

display(df.head(30))
print("Number of columns:", df.shape[1])

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.700,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.880,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.760,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.280,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.700,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
5,7.4,0.660,0.00,1.8,0.075,13.0,40.0,0.9978,3.51,0.56,9.4,5
6,7.9,0.600,0.06,1.6,0.069,15.0,59.0,0.9964,3.30,0.46,9.4,5
7,7.3,0.650,0.00,1.2,0.065,15.0,21.0,0.9946,3.39,0.47,10.0,7
8,7.8,0.580,0.02,2.0,0.073,9.0,18.0,0.9968,3.36,0.57,9.5,7
9,7.5,0.500,0.36,6.1,0.071,17.0,102.0,0.9978,3.35,0.80,10.5,5


Number of columns: 12


In [6]:
#exploring the dataset
print("Dataset shape:", df.shape)
print("\nMissing values:")
print(df.isna().sum())

Dataset shape: (1599, 12)

Missing values:
fixed acidity           0
volatile acidity        0
citric acid             0
residual sugar          0
chlorides               0
free sulfur dioxide     0
total sulfur dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
quality                 0
dtype: int64


There are no missing columns and there are a total of 1599 rows with 12 columns

The target is to predict the quality of the wine however, i will create another columns called the quality_class where by the quality will be categorized as either high, medium or low quality. This will then ensure that the quality of the wine will be expressed as a quality class.

In [7]:
# Remove rows with missing targets, if any then group the wines into classes. 0-4= low, 5-6=medium and 7-10=high quality
df = df.dropna(subset=["quality"]).copy()

df["quality_class"] = pd.cut(
    df["quality"],
    bins=[-1, 4, 6, 10],
    labels=[0, 1, 2]
).astype(int)

print("Class counts: 0 = Low, 1 = Medium, 2 = High")
print(df["quality_class"].value_counts().sort_index())

Class counts: 0 = Low, 1 = Medium, 2 = High
quality_class
0      63
1    1319
2     217
Name: count, dtype: int64


In total there are 63  low, 1319 medirm and 217 high

**split the dataset into training and testing sets in to 80% training, 20% testing**

In [8]:
X = df.drop(columns=["quality", "quality_class"])
y = df["quality_class"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Standardize the data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

Training shape: (1279, 11)
Testing shape: (320, 11)


 Out of the total 1599 rows, we slpit the data into 1279 training samples and 320 test samples. Note the wine dataset contains 11 wine chemicals  and our target (quality_class) contains 3 classes (low, medium and high). The data is then standardized to ensure that each feature has a mean of 0 and a standard deviation of 1, which helps improve model performance.

**Train logistic regression and random forest**

In [9]:
# Train logistic regression model
log_reg = LogisticRegression(max_iter=2000)
log_reg.fit(X_train, y_train)

# Predict on test set
y_pred = log_reg.predict(X_test)

# Evaluate performance
accuracy = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(
    y_test, y_pred, average="macro", zero_division=0  #Macro F1 gives each class equal importance
)
print(f"Logistic Regression Accuracy: {accuracy:.2f}")
print(f"Logistic Regression Macro F1-score: {macro_f1:.2f}")

Logistic Regression Accuracy: 0.84
Logistic Regression Macro F1-score: 0.47


In [10]:
# Train random forest classifier
random_forest = RandomForestClassifier(n_estimators=300)
random_forest.fit(X_train, y_train)

# Predict on test set
y_pred = random_forest.predict(X_test)

# Evaluate accuracy
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)

print(f"Random Forest Accuracy: {accuracy:.2f}")
print(f"Random Forest Macro F1-score: {f1:.2f}")


Random Forest Accuracy: 0.88
Random Forest Macro F1-score: 0.54


Comparing the logistic regression and random forest classifier.
For accuracy, the random forest correctly classifying 89% of test wines compared with 84% for logistic regression.
for F1-score (measures how well a model balances precision and recall with higher values indicating better performance (0-1). The random forest scored 55% while the logistic regression scored 47. meaning for both models the random forest performed better then the logistic regression.

Automate Model Selection Using AutoML using the Tpot model
**Install the Tpot**


In [11]:
%pip install tpot

Note: you may need to restart the kernel to use updated packages.


In [11]:
%pip install "setuptools<82"

Note: you may need to restart the kernel to use updated packages.


In [12]:
%pip install --force-reinstall "setuptools<82"

  Using cached setuptools-81.0.0-py3-none-any.whl.metadata (6.6 kB)
Using cached setuptools-81.0.0-py3-none-any.whl (1.1 MB)
Note: you may need to restart the kernel to use updated packages.


In [13]:
import sys
from importlib.metadata import version

print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("TPOT version:", version("tpot"))

Python executable: C:\Users\tebos\tpot-env\Scripts\python.exe
Python version: 3.13.16 (tags/v3.13.16:cbc944f, Sep 30 2026, 19:17:50) [MSC v.1944 64 bit (AMD64)]
TPOT version: 1.1.0


In [16]:
from tpot import TPOTClassifier
from sklearn.metrics import accuracy_score, f1_score

automl = TPOTClassifier(
    search_space="linear-light",
    scorers=["f1_macro"],
    scorers_weights=[1],
    max_time_mins=5,
    max_eval_time_mins=1,
    preprocessing=True,
    verbose=2,
    cv=3,
    n_jobs=1,
    random_state=42
)

# Train using your original, unscaled training features
automl.fit(X_train, y_train)

# Predict on test data
tpot_predictions = automl.predict(X_test)

# Evaluate performance
accuracy = accuracy_score(y_test, tpot_predictions)
macro_f1 = f1_score(
    y_test,
    tpot_predictions,
    average="macro",
    zero_division=0
)

print(f"TPOT Accuracy: {accuracy:.2f}")
print(f"TPOT Macro F1-score: {macro_f1:.2f}")

print("\nSelected pipeline:")
print(automl.fitted_pipeline_)

Generation: : 10it [05:00, 30.03s/it]


TPOT Accuracy: 0.83
TPOT Macro F1-score: 0.59

Selected pipeline:
Pipeline(steps=[('pipeline-1',
                 Pipeline(steps=[('impute_numeric', ColumnSimpleImputer())])),
                ('pipeline-2',
                 Pipeline(steps=[('minmaxscaler', MinMaxScaler()),
                                 ('selectpercentile',
                                  SelectPercentile(percentile=34.4292717112328)),
                                 ('featureunion-1',
                                  FeatureUnion(transformer_list=[('skiptransformer',
                                                                  SkipTransformer()),
                                                                 ('passthrough',
                                                                  Passthrough())])),
                                 ('featureunion-2',
                                  FeatureUnion(transformer_list=[('skiptransformer',
                                                                

Random forest achieved the highest accuracy, correctly classifying 89% of test wines. TPOT achieved the highest macro F1-score, suggesting a better balance of precision and recall across the low-, medium-, and high-quality classes, despite its lower accuracy.
TPOT performed best by macro F1, while random forest performed best by overall accuracy. 